In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#data load
loader = TextLoader("speech.txt")
text_documents = loader.load()

# data transform
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 30)
final_docs = text_splitter.split_documents(text_documents)

#embedding 
## Bash cmd - ollama pull nomic-embed-text =>great for RAG pipelines
embeddings = OllamaEmbeddings(model="nomic-embed-text")

#vector store with doc embeddings

db = Chroma.from_documents(final_docs,embeddings)
db

C:\Users\HP\AppData\Local\Temp\ipykernel_19356\2251333106.py:11: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [3]:
#querying
query = "What is the speaker's attitude toward the German people?"
result = db.similarity_search(query)
result[0].page_content

'It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early'

In [4]:
#saving to the local 
db = Chroma.from_documents(final_docs,embeddings,persist_directory="./chroma_db")

In [7]:
#load the model 
new_db = Chroma(persist_directory='./chroma_db',embedding_function=embeddings)
new_db.similarity_search(query)

[Document(id='e0a6d81e-3b81-406b-92da-8bb8b7f10948', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early'),
 Document(id='a84461cc-3e72-47b5-a83e-969d84c3d8b9', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendshipâ€”exercising a patience and forbearance which would otherwise have been impossible. We shall, happily, still have an opportunity to prove that friendship in our daily attitude and actions tow

In [6]:
##retriever 

retreiver = db.as_retriever()
retreiver.invoke(query)

[Document(id='e0a6d81e-3b81-406b-92da-8bb8b7f10948', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early'),
 Document(id='a84461cc-3e72-47b5-a83e-969d84c3d8b9', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendshipâ€”exercising a patience and forbearance which would otherwise have been impossible. We shall, happily, still have an opportunity to prove that friendship in our daily attitude and actions tow